# SceneTIQ backend on Colab

Starts the SceneTIQ API on this Colab GPU, stores every Scene on the runtime's local disk, and exposes the API at a stable URL via a Cloudflare **named tunnel** (ADR 0004).

**Before the first run** (see `README.md`):
1. Add Colab secrets (🔑 sidebar): `SCENETIQ_TOKEN` (any long random string) and `CLOUDFLARE_TUNNEL_TOKEN` (from your named tunnel).
2. In the Cloudflare dashboard, route the tunnel's public hostname to `http://localhost:8001`.
3. Runtime → Change runtime type → GPU (A100 80GB).

Run all cells. **Scenes last only as long as this runtime:** when it disconnects or is deleted, they are gone and must be uploaded again. The URL stays the same. If only the server restarts, Scenes remain and jobs that were running are marked failed. Re-running the server cell replaces the running server, so it also stops any job in progress.

The last cell keeps running and shows the API log, with a progress bar for the photo upload and the job being processed. Logs are not saved anywhere else. With the real Pipeline, the first job of a session stays on `reconstructing` for several minutes while the models (about 15 GB) download.

In [ ]:
# Parameters
REPO_URL = "https://github.com/luatducvo/scenetiq.git"
BRANCH = "main"
PORT = 8001

# Wins over src/scenetiq/config.py, which lists every setting. A setting left out here uses its default there.
CONFIG = {
    "storage_root": "/content/scenetiq-data",    # Scenes + SQLite: local disk, lost with the runtime
    "pipeline": "real",                          # "real" (GPU model stack) or "fake" (no models, for testing)
    "view_stride": 1,                            # Reconstruct every Nth uploaded photo
    "min_views": 1,                              # Fewest photos a Capture may upload
    "max_views": 2000,                           # Most photos a Capture may upload
    "da3": {"process_res": 504},
}

In [ ]:
import subprocess, sys, pathlib
# The clone's name must not be a valid module name: Colab's cwd (/content) is on sys.path,
# and a folder called "scenetiq" there would shadow the real package as an empty namespace.
repo = pathlib.Path("/content/scenetiq-repo")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
# scenetiq.colab needs only the standard library, so it is importable straight from the clone.
sys.path.insert(0, str(repo / "src"))
from scenetiq import colab
colab.install_packages(repo, colab.configured_pipeline(CONFIG), colab.live_display())

In [ ]:
from google.colab import userdata
from scenetiq import colab

token = userdata.get("SCENETIQ_TOKEN")
tunnel_token = userdata.get("CLOUDFLARE_TUNNEL_TOKEN")

env = colab.server_environment(token, CONFIG)
server = colab.start_server(env, PORT)
colab.wait_until_ready(server, PORT, token, timeout=600)
print("backend ready on port", PORT)

In [ ]:
tunnel = colab.start_named_tunnel(tunnel_token)
print("tunnel started; set VITE_BACKEND_URL in frontend/.env to your tunnel hostname, then open the frontend")

In [ ]:
# API log, with upload and job progress. Keeps running; stop it to run another cell, the backend keeps going.
colab.watch(server)